In [1]:
pip install numpy pandas matplotlib seaborn scikit-learn keras tensorflow 

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#####                      KNN 
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################




import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Function to load images from a directory
def load_images_from_folder(folder, max_samples=234):
    images = []
    labels = []
    for fabric_type in os.listdir(folder):
        fabric_path = os.path.join(folder, fabric_type)
        if os.path.isdir(fabric_path):
            sample_count = 0
            for filename in os.listdir(fabric_path):
                if filename.endswith(('.png', '.jpg', '.jpeg', 'gif')) and sample_count < max_samples:
                    img = Image.open(os.path.join(fabric_path, filename)).convert('RGB')
                    img = img.resize((100, 100))  # Resize images to match VGG16 input
                    img_array = np.array(img) / 255.0
                    images.append(img_array)
                    labels.append(fabric_type)
                    sample_count += 1
    return np.array(images), np.array(labels)

# Load images and labels
IMAGE_FOLDER = r"C:\Users\Henry Austin\Desktop\comp1012 project\SparseReflectance-Maps-for-Fabric-Characterization"
X, y = load_images_from_folder(IMAGE_FOLDER, max_samples=234)

# Print shapes and unique labels
print("Loaded images shape:", X.shape)
print("Loaded labels shape:", y.shape)
print("Unique labels:", np.unique(y))

# Encode labels to numbers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=194)

# Load pre-trained VGG16 model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(100, 100, 3))

# Freeze all convolutional layers
for layer in base_model.layers:
    layer.trainable = False

# Create a new model using the base model as a feature extractor and flatten the output
feature_extractor_model = Model(inputs=base_model.input, outputs=Flatten()(base_model.output))

# Extract features using the CNN
features_train = feature_extractor_model.predict(X_train)
features_test = feature_extractor_model.predict(X_test)

# Train KNN model
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(features_train, y_train)

# Evaluate KNN model
predictions_knn = knn.predict(features_test)
accuracy_knn = accuracy_score(y_test, predictions_knn)
print(f'KNN Accuracy: {accuracy_knn:.2f}')

# Print classification report
print("KNN Classification Report:\n", classification_report(y_test, predictions_knn))

# Confusion matrix
cm = confusion_matrix(y_test, predictions_knn)
np.set_printoptions(precision=2)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix for KNN')
plt.colorbar()
tick_marks = np.arange(len(np.unique(y_encoded)))
plt.xticks(tick_marks, np.unique(y_encoded), rotation=45)
plt.yticks(tick_marks, np.unique(y_encoded))
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()

# Print the numbers inside the confusion matrix
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")

# Save the confusion matrix figure
plt.savefig('confusion_matrix_knn.png')
plt.show()

In [ ]:
#####                 Logistic regression

###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################




# Initialize classifiers  
models = {
    'Logistic Regression': LogisticRegression(max_iter=10),  # Increased max_iter  
    'SVM': SVC(probability=True)  # Enable probability estimates for SVM
}
# Initialize classifiers  
models = {
    'Logistic Regression': LogisticRegression(max_iter=10),  # Increased max_iter  
    'SVM': SVC(probability=True)  # Enable probability estimates for SVM
}

# Train and evaluate each model  
accuracies = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)  # Train the model  
    predictions = model.predict(X_test)  # Make predictions  
    accuracy = accuracy_score(y_test, predictions)  # Calculate accuracy  
    accuracies[model_name] = accuracy  # Store accuracy  
    print(f'{model_name} Accuracy: {accuracy:.2f}')

    # Print classification report  
    print(f"{model_name} Classification Report:\n", classification_report(y_test, predictions))
    
    # Confusion Matrix  
    cm = confusion_matrix(y_test, predictions)
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix for {model_name}')
    plt.colorbar()
    tick_marks = np.arange(len(np.unique(y_encoded)))
    plt.xticks(tick_marks, np.unique(y_encoded), rotation=45)
    plt.yticks(tick_marks, np.unique(y_encoded))
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

    # Print the numbers inside the confusion matrix  
    thresh = cm.max() / 2.  # Threshold for color selection  
    for i in range(cm.shape[0]):  # Iterate over rows  
        for j in range(cm.shape[1]):  # Iterate over columns  
            plt.text(j, i, format(cm[i, j], 'd'),  # Print the number  
                     ha="center", va="center", 
                     color="white" if cm[i, j] > thresh else "black")

    # Save the confusion matrix figure  
    plt.savefig(f'confusion_matrix_{model_name}.png')  # Change the path or filename as needed  
    plt.show()  # Show the plot after saving

# Additional Plots

# 1. ROC Curve for SVM  
n_classes = len(np.unique(y_encoded))
y_test_binarized = label_binarize(y_test, classes=np.arange(n_classes))  # Binarize the output  
y_pred_prob = models['SVM'].predict_proba(X_test)  # Get probabilities

# Plot ROC curves for each class  
plt.figure(figsize=(10, 8))
colors = ['blue', 'red', 'green', 'orange', 'purple', 'cyan']
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_binarized[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2, label=f'ROC curve for class {label_encoder.inverse_transform([i])[0]} (area = {roc_auc:0.2f})')

# Plot ROC curve for the micro-average  
fpr_micro, tpr_micro, _ = roc_curve(y_test_binarized.ravel(), y_pred_prob.ravel())
roc_auc_micro = auc(fpr_micro, tpr_micro)
plt.plot(fpr_micro, tpr_micro, color='grey', linestyle='--', lw=2, label='Micro-average ROC curve (area = {0:0.2f})'.format(roc_auc_micro))

plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multiclass ROC Curves')
plt.grid()  # Add grid lines  
plt.legend(loc='lower right')
plt.savefig('multiclass_roc_curves.png')
plt.show()  # Show the ROC curve without blocking

In [ ]:
#####                  random forest


###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################



import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Function to load images from a directory  
def load_images_from_folder(folder, max_samples=234):  # Adjust max_samples for quick testing  
    images = []
    labels = []
    for fabric_type in os.listdir(folder):
        fabric_path = os.path.join(folder, fabric_type)
        if os.path.isdir(fabric_path):
            sample_count = 0  # To limit the number of samples  
            for filename in os.listdir(fabric_path):
                if filename.endswith(('.png', '.jpg', '.jpeg', 'gif')) and sample_count < max_samples:
                    img = Image.open(os.path.join(fabric_path, filename)).convert('RGB')
                    img = img.resize((50, 50))  # Resize images to a smaller size  
                    img_array = np.array(img) / 255.0  # Normalize image data  
                    images.append(img_array)
                    labels.append(fabric_type)
                    sample_count += 1  
    return np.array(images, dtype=np.float32), np.array(labels)  # Use float32 to reduce memory usage

# Load images and labels  
IMAGE_FOLDER = r"C:\Users\Henry Austin\Desktop\comp1012 project\SparseReflectance-Maps-for-Fabric-Characterization"  # Update this path  
X, y = load_images_from_folder(IMAGE_FOLDER, max_samples=234)  # Adjust max_samples for quick testing

# Print shapes and unique labels  
print("Loaded images shape:", X.shape)
print("Loaded labels shape:", y.shape)
print("Unique labels:", np.unique(y))

# Encode labels to numbers  
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Scale the data  
scaler = StandardScaler()
X_reshaped = X.reshape(X.shape[0], -1)  # Flatten the images for scaling  
X_scaled = scaler.fit_transform(X_reshaped)  # Fit and transform the data

# Split the dataset into training and testing sets  
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=199)

# Set up the parameters for grid search
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

# Initialize Random Forest classifier with grid search
random_forest_model = GridSearchCV(estimator=RandomForestClassifier(random_state=42), param_grid=param_grid, cv=5, n_jobs=1, verbose=2)  # Reduce parallel processing

# Train the model  
random_forest_model.fit(X_train, y_train)  # Train the model  
print("Best parameters found: ", random_forest_model.best_params_)

# Use the best model to make predictions  
best_rf = random_forest_model.best_estimator_
predictions = best_rf.predict(X_test)  # Make predictions  
accuracy = accuracy_score(y_test, predictions)  # Calculate accuracy  
print(f'Optimized Random Forest Accuracy: {accuracy:.2f}')

# Print classification report  
print("Optimized Random Forest Classification Report:\n", classification_report(y_test, predictions))

# Confusion Matrix  
cm = confusion_matrix(y_test, predictions)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix for Optimized Random Forest')
plt.colorbar()
tick_marks = np.arange(len(np.unique(y_encoded)))
plt.xticks(tick_marks, np.unique(y_encoded), rotation=45)
plt.yticks(tick_marks, np.unique(y_encoded))
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()

# Print the numbers inside the confusion matrix  
thresh = cm.max() / 2.  # Threshold for color selection  
for i in range(cm.shape[0]):  
    for j in range(cm.shape[1]):  
        plt.text(j, i, format(cm[i, j], 'd'),  
                 ha="center", va="center", 
                 color="white" if cm[i, j] > thresh else "black")

# Save the confusion matrix figure  
plt.savefig('confusion_matrix_optimized_random_forest.png')  # Change the path or filename as needed  
plt.show()  # Show the plot after saving

In [ ]:
#####                      SVM



###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################




import os  
import pandas as pd
import numpy as np  
import seaborn as sns  
import matplotlib.pyplot as plt  
from PIL import Image  
from sklearn.model_selection import train_test_split  
from sklearn.neighbors import KNeighborsClassifier  
from sklearn.tree import DecisionTreeClassifier  
from sklearn.ensemble import RandomForestClassifier  
from sklearn.svm import SVC  
from sklearn.linear_model import LogisticRegression  
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc  
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize  
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Function to load images from a directory  
def load_images_from_folder(folder, max_samples=234):  # Adjust max_samples for quick testing  
    images = []
    labels = []
    for fabric_type in os.listdir(folder):
        fabric_path = os.path.join(folder, fabric_type)
        if os.path.isdir(fabric_path):
            sample_count = 0  # To limit the number of samples  
            for filename in os.listdir(fabric_path):
                if filename.endswith(('.png', '.jpg', '.jpeg', 'gif')) and sample_count < max_samples:
                    img = Image.open(os.path.join(fabric_path, filename)).convert('RGB')
                    img = img.resize((100, 100))  # Resize images to a smaller size  
                    #img = img.resize((224, 224))  # Resize images to a larger size
                    img_array = np.array(img) / 255.0  # Normalize image data  
                    images.append(img_array)
                    
                    labels.append(fabric_type)
                    sample_count += 1  
    return np.array(images), np.array(labels)

# Load images and labels  
IMAGE_FOLDER = r"C:\Users\Henry Austin\Desktop\comp1012 project\SparseReflectance-Maps-for-Fabric-Characterization"  # Update this path  
X, y = load_images_from_folder(IMAGE_FOLDER, max_samples=234)  # Adjust max_samples for quick testing

# Print shapes and unique labels  
print("Loaded images shape:", X.shape)
print("Loaded labels shape:", y.shape)
print("Unique labels:", np.unique(y))

# Encode labels to numbers  
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Scale the data  
scaler = StandardScaler()
X_reshaped = X.reshape(X.shape[0], -1)  # Flatten the images for scaling  
X_scaled = scaler.fit_transform(X_reshaped)  # Fit and transform the data

# Split the dataset into training and testing sets  
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=194)

# Initialize classifiers  
models = {
   
    'SVM': SVC(kernel='rbf', C=5.5, probability=True),  
    
}

# Train and evaluate each model  
accuracies = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)  # Train the model  
    predictions = model.predict(X_test)  # Make predictions  
    accuracy = accuracy_score(y_test, predictions)  # Calculate accuracy  
    accuracies[model_name] = accuracy  # Store accuracy  
    print(f'{model_name} Accuracy: {accuracy:.2f}')

    # Print classification report  
    print(f"{model_name} Classification Report:\n", classification_report(y_test, predictions))
    
    # Confusion Matrix  
    cm = confusion_matrix(y_test, predictions)
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix for {model_name}')
    plt.colorbar()
    tick_marks = np.arange(len(np.unique(y_encoded)))
    plt.xticks(tick_marks, np.unique(y_encoded), rotation=45)
    plt.yticks(tick_marks, np.unique(y_encoded))
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

    # Print the numbers inside the confusion matrix  
    thresh = cm.max() / 2.  # Threshold for color selection  
    for i in range(cm.shape[0]):  # Iterate over rows  
        for j in range(cm.shape[1]):  # Iterate over columns  
            plt.text(j, i, format(cm[i, j], 'd'),  # Print the number  
                     ha="center", va="center", 
                     color="white" if cm[i, j] > thresh else "black")

    # Save the confusion matrix figure  
    plt.savefig(f'confusion_matrix_{model_name}.png')  # Change the path or filename as needed  
    plt.show()  # Show the plot after saving  
    
# Train the VGG16 Model  
X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=194)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(100, 100, 3 ))

for layer in base_model.layers:
    layer.trainable = False

vgg16_model = Sequential()
vgg16_model.add(base_model)
vgg16_model.add(Flatten())
vgg16_model.add(Dense(256, activation='relu'))
vgg16_model.add(Dropout(0.5))
vgg16_model.add(Dense(len(np.unique(y_encoded)), activation='softmax'))

vgg16_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Data augmentation  
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Fit the VGG16 model with data augmentation  
history = vgg16_model.fit(datagen.flow(X_full_train.reshape(-1, 100, 100, 3), y_full_train, batch_size=16), 
                           epochs=7, validation_data=(X_full_test.reshape(-1, 100, 100, 3), y_full_test))

# Evaluate the VGG16 model  
test_loss, test_accuracy = vgg16_model.evaluate(X_full_test.reshape(-1, 100, 100, 3), y_full_test)
print(f'Test Accuracy (VGG16): {test_accuracy:.2f}')

# Create a figure for the combined plot  
plt.figure(figsize=(12, 6))

# Plot training and validation loss  
plt.subplot(1, 1, 1)
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.title('Model Loss & Accuracy')
plt.legend(loc='upper left')
plt.grid()

# Create a secondary y-axis for accuracy  
ax2 = plt.gca().twinx()
ax2.plot(history.history['accuracy'], label='Training Accuracy', color='green')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', color='red')
ax2.set_ylabel('Accuracy')
ax2.legend(loc='upper right')

plt.show()

# Additional Plots

# 1. ROC Curve for SVM  
n_classes = len(np.unique(y_encoded))
y_test_binarized = label_binarize(y_test, classes=np.arange(n_classes))  # Binarize the output  
y_pred_prob = models['SVM'].predict_proba(X_test)  # Get probabilities

# Plot ROC curves for each class  
plt.figure(figsize=(10, 8))
colors = ['blue', 'red', 'green', 'orange', 'purple', 'cyan']
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_binarized[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2, label='ROC curve for class {0} (area = {1:0.2f})'.format(label_encoder.inverse_transform([i])[0], roc_auc))

# Plot ROC curve for the micro-average  
fpr_micro, tpr_micro, _ = roc_curve(y_test_binarized.ravel(), y_pred_prob.ravel())
roc_auc_micro = auc(fpr_micro, tpr_micro)
plt.plot(fpr_micro, tpr_micro, color='grey', linestyle='--', lw=2, label='Micro-average ROC curve (area = {0:0.2f})'.format(roc_auc_micro))

plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multiclass ROC Curves')
plt.grid()  # Add grid lines  
plt.legend(loc='lower right')
plt.savefig('multiclass_roc_curves.png')
plt.show()  # Show the ROC curve without blocking

# Initialize a dictionary to hold classification reports  
classification_reports = {}

# Train and evaluate each model  
for model_name, model in models.items():
    model.fit(X_train, y_train)  # Train the model  
    predictions = model.predict(X_test)  # Make predictions  
    accuracy = accuracy_score(y_test, predictions)  # Calculate accuracy  
    print(f'{model_name} Accuracy: {accuracy:.2f}')

    # Generate classification report  
    report = classification_report(y_test, predictions, target_names=label_encoder.classes_, output_dict=True)
    classification_reports[model_name] = report  # Store the classification report for plotting

# Plotting classification reports as multi-color heatmaps  
for model_name, report in classification_reports.items():
    # Convert the classification report to a DataFrame  
    report_df = pd.DataFrame(report).transpose()
    
    # Set up the matplotlib figure  
    plt.figure(figsize=(10, 6))
    
    # Create a multi-color heatmap for precision, recall, and F1-score  
    sns.heatmap(report_df[['precision', 'recall', 'f1-score']], annot=True, fmt=".2f", cmap='viridis', cbar=True)
    
    # Set title and labels  
    plt.title(f'Classification Report Heatmap for {model_name}', fontsize=16)
    plt.ylabel('Classes')
    plt.xlabel('Metrics')
    
    # Show the plot  
    plt.tight_layout()
    plt.show()

In [ ]:
#####                  decision tree




###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################






import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.utils import class_weight
from sklearn import metrics
import matplotlib.pyplot as plt

# Function to load images from a directory  
def load_images_from_folder(folder, max_samples=235):  
    images = []
    labels = []
    for fabric_type in os.listdir(folder):
        fabric_path = os.path.join(folder, fabric_type)
        if os.path.isdir(fabric_path):
            sample_count = 0  
            for filename in os.listdir(fabric_path):
                if filename.endswith(('.png', '.jpg', '.jpeg', 'gif')) and sample_count < max_samples:
                    img = Image.open(os.path.join(fabric_path, filename)).convert('RGB')
                    img = img.resize((100, 100))  
                    img_array = np.array(img) / 255.0  
                    images.append(img_array.flatten())  # Flatten the image
                    labels.append(fabric_type)
                    sample_count += 1  
    return np.array(images), np.array(labels)

# Load images and labels  
IMAGE_FOLDER = r"C:\Users\Henry Austin\Desktop\comp1012 project\SparseReflectance-Maps-for-Fabric-Characterization"
X, y = load_images_from_folder(IMAGE_FOLDER, max_samples=235)

# Print shapes and unique labels  
print("Loaded images shape:", X.shape)
print("Loaded labels shape:", y.shape)
print("Unique labels:", np.unique(y))

# Encode labels to numbers  
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Scale the data  
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)  # Fit and transform the data

# Split the dataset into training and testing sets  
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=142)

# Initialize Decision Tree classifier  
dt = DecisionTreeClassifier(random_state=42)

# Set up the parameters for grid search
param_grid = {
    'max_depth': [None, 5,10,15, 20,25, 30],
    'min_samples_split': [2, 5,8, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': [None, 'balanced']  # Use 'balanced' for class weight adjustment
}

# Create grid search object
grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# Fit the model
grid_search.fit(X_train, y_train)

# Print best parameters
print("Best parameters:", grid_search.best_params_)

# Use the best model to make predictions
best_dt = grid_search.best_estimator_
predictions = best_dt.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f'Optimized Decision Tree Accuracy: {accuracy:.2f}')

# Print classification report
print("Optimized Decision Tree Classification Report:\n", classification_report(y_test, predictions))

# Confusion Matrix
cm = confusion_matrix(y_test, predictions)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix for Optimized Decision Tree')
plt.colorbar()
tick_marks = np.arange(len(np.unique(y_encoded)))
plt.xticks(tick_marks, np.unique(y_encoded), rotation=45)
plt.yticks(tick_marks, np.unique(y_encoded))
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()

# Print the numbers inside the confusion matrix
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")

# Save the confusion matrix figure
plt.savefig('confusion_matrix_optimized_decision_tree.png')
plt.show()

In [ ]:
#####                      CNN





###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################
###############please run the code separately##################





import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Enable memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# Function to load images from a folder
def load_images_from_folder(folder, target_size=(100, 100), max_samples_per_class=200):
    images = []
    labels = []
    for class_name in os.listdir(folder):
        class_path = os.path.join(folder, class_name)
        if os.path.isdir(class_path):
            image_files = os.listdir(class_path)[:max_samples_per_class]
            for image_name in image_files:
                if image_name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif')):
                    image_path = os.path.join(class_path, image_name)
                    try:
                        img = Image.open(image_path).convert('RGB')
                        img = img.resize(target_size)
                        img_array = np.array(img) / 255.0
                        images.append(img_array)
                        labels.append(class_name)
                    except Exception as e:
                        print(f'Error loading image {image_path}: {e}')
    return np.array(images), np.array(labels)

# Load images and labels
IMAGE_FOLDER = r"C:\Users\Henry Austin\Desktop\comp1012 project\SparseReflectance-Maps-for-Fabric-Characterization"
X, y = load_images_from_folder(IMAGE_FOLDER, target_size=(100, 100), max_samples_per_class=200)

# Check if data is loaded successfully
print(f'Total images loaded: {len(X)}')

# Encode labels to numerical values
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Print the sizes of the training and testing sets
print(f'Number of training samples: {len(X_train)}')
print(f'Number of testing samples: {len(X_test)}')

# Define the transfer learning model using MobileNetV2
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(100, 100, 3))

# Freeze the convolutional layers of the pre-trained model
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

# Split the training set into training and validation sets
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y_encoded, test_size=0.1, random_state=42, stratify=y_encoded
)
num_classes = len(np.unique(y_encoded))
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Print the model summary
model.summary()

# Define callback functions
early_stopping = EarlyStopping(
    monitor='val_loss', patience=10, verbose=1, restore_best_weights=True
)
model_checkpoint = ModelCheckpoint(
    'best_transfer_learning_model.keras', monitor='val_loss', save_best_only=True
)

# Train the model with a reduced batch size
history = model.fit(
    X_train, y_train,
    epochs=80,
    batch_size=16,  # Reduced batch size
    validation_data=(X_test, y_test),
    callbacks=[early_stopping, model_checkpoint]
)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

# Visualize the training process
import matplotlib.pyplot as plt

# Plot accuracy curves
plt.figure(figsize=(8, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Plot loss curves
plt.figure(figsize=(8, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()